# 05 - Demo: from audio to notes & chords

This final notebook wraps the whole project in a small **desktop window** (built
with Tkinter, part of the Python standard library). Pick a sound and it runs the
full pipeline from the previous notebooks:

- detected **pitch** with both methods (autocorrelation and HPS),
- the dominant **chord** from the chromagram,
- the **waveform**, **spectrogram** and **chromagram**.

In [1]:
import sys
sys.path.append('../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt

import common as C
import stft as ST
import pitch as P
import chroma as CH

## 1. The analysis, in one function

`analyze_core` takes a signal and returns the detected pitch (both methods) and
the dominant chord. This is the same pipeline used throughout the project.

In [2]:
def analyze_core(signal, Fs):
    # pitch on a short segment from the middle of the clip
    middle = len(signal) // 2
    seg = signal[middle:middle + int(0.3 * Fs)]
    f_ac = P.detect_pitch_autocorrelation(seg, Fs, fmin=80, fmax=1200)
    f_hps = P.detect_pitch_hps(seg, Fs, num_harmonics=5, fmin=80, fmax=1200)

    # dominant chord from the average chromagram
    times, H = CH.chromagram(signal, Fs)
    chord, sim = CH.detect_chord(H.mean(axis=1))
    return f_ac, f_hps, chord, sim, times, H


def note_str(f):
    if not f:
        return '-'
    name, octave, _ = C.freq_to_note(f)
    return f'{name}{octave}  ({f:.0f} Hz)'

## 2. Quick static check

Run it once on `piano.wav` so the notebook shows a result even without opening
the window.

In [3]:
Fs, piano = C.load_wav('../data/piano.wav')
f_ac, f_hps, chord, sim, times, H = analyze_core(piano, Fs)

print('Autocorrelation:', note_str(f_ac))
print('HPS:            ', note_str(f_hps))
print('Dominant chord: ', chord, '(similarity %.2f)' % sim)

Autocorrelation: G2  (97 Hz)
HPS:             A3  (220 Hz)
Dominant chord:  A:min (similarity 0.74)


## 3. The pink window

Run the cell below to open the demo window. Click one of the example buttons
(**Piano**, **Guitar**, **A minor**, **C major**) or **Choose .wav...** to load
your own recording. 

The result and the three plots update instantly.

In [6]:
import tkinter as tk
from tkinter import filedialog
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg


BG, CARD, HOT, DEEP, PURP = '#ffe6f2', '#fff0f7', '#ff8fc0', '#d6478a', '#a86bd4'


def open_demo():
    root = tk.Tk()
    root.title('Music Analysis')
    root.configure(bg=BG)

    tk.Label(root, text='music analysis', bg=BG, fg=DEEP,
             font=('DejaVu Sans', 22, 'bold')).pack(pady=(14, 0))
    tk.Label(root, text='from audio to notes & chords', bg=BG, fg=PURP,
             font=('DejaVu Sans', 11)).pack()

    result = tk.Label(root, text='Pick a sound below', bg=CARD, fg=DEEP,
                      font=('DejaVu Sans', 14), width=52, height=4, justify='center')
    result.pack(pady=12, padx=16)

    fig = Figure(figsize=(6.2, 5.2), facecolor=BG)
    ax1, ax2, ax3 = fig.add_subplot(311), fig.add_subplot(312), fig.add_subplot(313)
    fig.subplots_adjust(hspace=0.9, left=0.12, right=0.97, top=0.93, bottom=0.1)
    canvas = FigureCanvasTkAgg(fig, master=root)
    canvas.get_tk_widget().pack(padx=16, pady=(0, 10))

    def show(signal, Fs, label):
        f_ac, f_hps, chord, sim, times, H = analyze_core(signal, Fs)
        result.config(text=(f'Autocorrelation:  {note_str(f_ac)}\n'
                            f'HPS:  {note_str(f_hps)}\n'
                            f'Chord:  {chord}   (similarity {sim:.2f})'))
        for ax in (ax1, ax2, ax3):
            ax.clear()
        t = np.arange(len(signal)) / Fs
        ax1.plot(t, signal, color=HOT, linewidth=0.6)
        ax1.set_title(f'{label} - waveform', color=DEEP, fontsize=9)
        f, tt, mag = ST.spectrogram(signal, Fs, window_size=4096)
        ax2.pcolormesh(tt, f, mag, shading='auto', cmap='magma'); ax2.set_ylim(0, 2000)
        ax2.set_title('spectrogram', color=DEEP, fontsize=9)
        ax3.imshow(H, aspect='auto', origin='lower', cmap='RdPu',
                   extent=[times[0], times[-1], -0.5, 11.5])
        ax3.set_yticks(range(0, 12, 2)); ax3.set_yticklabels(C.NOTE_NAMES[::2], fontsize=7)
        ax3.set_title('chromagram', color=DEEP, fontsize=9)
        canvas.draw()

    def load_example(name):
        if name == 'A minor':
            sig = C.generate_chord([C.note_to_freq(m) for m in [57, 60, 64]], 2.0, 44100, num_harmonics=4)
            show(sig, 44100, 'A minor')
        elif name == 'C major':
            sig = C.generate_chord([C.note_to_freq(m) for m in [60, 64, 67]], 2.0, 44100, num_harmonics=4)
            show(sig, 44100, 'C major')
        else:
            fs, sig = C.load_wav(f'../data/{name}')
            show(sig, fs, name)

    def choose_file():
        path = filedialog.askopenfilename(filetypes=[('WAV files', '*.wav')])
        if path:
            fs, sig = C.load_wav(path)
            show(sig, fs, path.split('/')[-1])

    row = tk.Frame(root, bg=BG); row.pack(pady=(0, 16))

    def mkbtn(text, cmd):
        return tk.Button(row, text=text, command=cmd, bg=HOT, fg='white',
                         font=('DejaVu Sans', 11, 'bold'), relief='flat',
                         activebackground=DEEP, activeforeground='white',
                         padx=12, pady=6, bd=0, cursor='hand2')

    for txt, name in [('Piano', 'piano.wav'), ('Guitar', 'guitar.wav'),
                      ('A minor', 'A minor'), ('C major', 'C major')]:
        mkbtn(txt, lambda n=name: load_example(n)).pack(side='left', padx=5)
    mkbtn('Choose .wav...', choose_file).pack(side='left', padx=5)

    root.mainloop()


open_demo()